In [2]:
import os
import cv2
import shutil
import numpy as np


def aplicar_augmentacion(img):
    # 1. Brillo y contraste aleatorio
    alpha = np.random.uniform(0.7, 1.3)
    beta = np.random.randint(-30, 30)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    # 2. Flip horizontal aleatorio
    hubo_flip = False
    if np.random.rand() > 0.5:
        img = cv2.flip(img, 1)
        hubo_flip = True

    # 3. Ruido gaussiano leve
    ruido = np.random.normal(0, 8, img.shape).astype(np.uint8)
    img = cv2.add(img, ruido)

    # 4. Pequeña rotación
    h, w = img.shape[:2]
    angulo = np.random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angulo, 1.0)
    img = cv2.warpAffine(img, M, (w, h))

    return img, hubo_flip


def copiar_label_con_flip(lbl_path, dest_path, hubo_flip):
    with open(lbl_path, "r") as f:
        lineas = f.readlines()

    nuevas_lineas = []
    for linea in lineas:
        partes = linea.strip().split()
        if not partes:
            continue
        clase = partes[0]
        valores = list(map(float, partes[1:]))

        if hubo_flip:
            # Bounding box normal: clase x y w h
            if len(valores) == 4:
                valores[0] = 1 - valores[0]  # invertir x_center

            # Segmentación: clase x1 y1 x2 y2 x3 y3 ...
            else:
                for j in range(0, len(valores), 2):  # cada x está en posición par
                    valores[j] = 1 - valores[j]

        linea_nueva = clase + " " + " ".join(f"{v:.6f}" for v in valores) + "\n"
        nuevas_lineas.append(linea_nueva)

    with open(dest_path, "w") as f:
        f.writelines(nuevas_lineas)


def oversample_con_augmentacion(images_dir, labels_dir, clase_id=1, multiplicador=2):
    archivos = []
    for txt in os.listdir(labels_dir):
        if not txt.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, txt)) as f:
            clases = [int(l.split()[0]) for l in f if l.strip()]
        if clase_id in clases:
            archivos.append(txt.replace(".txt", ""))

    print(f"Imágenes con clase {clase_id}: {len(archivos)}")

    for nombre in archivos:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            src = os.path.join(images_dir, nombre + ext)
            if os.path.exists(src):
                img_path = src
                img_ext = ext
                break
        if img_path is None:
            continue

        img = cv2.imread(img_path)
        lbl_path = os.path.join(labels_dir, nombre + ".txt")

        for i in range(multiplicador - 1):
            augmentada, hubo_flip = aplicar_augmentacion(img)
            sufijo = f"_os{i+1}"

            cv2.imwrite(os.path.join(images_dir, nombre + sufijo + img_ext), augmentada)

            copiar_label_con_flip(
                lbl_path,
                os.path.join(labels_dir, nombre + sufijo + ".txt"),
                hubo_flip
            )

    print(f"✅ Listo. Copias generadas: {len(archivos) * (multiplicador - 1)}")


# ▶️ EJECUTAR
oversample_con_augmentacion(
    images_dir="dataset/train/images",
    labels_dir="dataset/train/labels",
    clase_id=1,        # clase "robo"
    multiplicador=3    # genera 2 copias extra por imagen → triplica
)

Imágenes con clase 1: 775
✅ Listo. Copias generadas: 1550
